In [2]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('../database/Reddit_Data.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [ ]:
print(df.info())
print("Target data distribution: ",[int((i/37249)*100) for i in df.category.value_counts()])

<class 'pandas.DataFrame'>
RangeIndex: 37249 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   clean_comment  37149 non-null  str  
 1   category       37249 non-null  int64
dtypes: int64(1), str(1)
memory usage: 582.1 KB
None
Target data distribution:  [42, 35, 22]


#### Contain some (few 100) Null values--> Will drop them

In [5]:
df.dropna(inplace=True)
print(df.isnull().sum())
df.info()

clean_comment    0
category         0
dtype: int64
<class 'pandas.DataFrame'>
Index: 37149 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   clean_comment  37149 non-null  str  
 1   category       37149 non-null  int64
dtypes: int64(1), str(1)
memory usage: 870.7 KB


#### Preprocessing like - (Text cleanup | Basic Preprocessing | Advance Preprocessing)

##### Basic  Preprocessing
(always used)- tokenization, 

(optionaly used)- 
1. stop words like(is, am, the, etc..) removal
2. stemming (words having same meaning with slight variation like dancing, dance) replace   them with same word
3. lower casing, digit, punchuation/exclaimatory/signs word removal
4. replacing short words with their full form, eg: ASAP = as soon as possible
5. spelling mistakes

##### Advance Preprocessing
1. POS tagging- Parts of speech (noun, pronoun, verb etc..) tagging
2. Parsing
3. Co-reference resolution (reference memory like ram is good boy, he is 10. ram = he (co-refrence))

In [6]:
#lower casing
df['clean_comment'] = df['clean_comment'].str.lower()

In [7]:
#remove punctuation
import string
exclude = string.punctuation
print(exclude)
def removepunc(text:str):
    return text.translate(str.maketrans('','',exclude))

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [8]:
df['clean_comment'] = df['clean_comment'].apply(removepunc)
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [9]:
#stop word removal
from nltk.corpus import stopwords
import time
en_stopwords = stopwords.words('english')

def removestopwords(text:str):
    newword = []
    for word in text.split():
        if word not in en_stopwords:
            newword.append(word)
    x = newword[:]
    newword.clear()
    return " ".join(x)

df['clean_comment'] = df.clean_comment.apply(removestopwords)


In [10]:
#emoji replacement with their meaning
import emoji
df.clean_comment = df.clean_comment.apply(emoji.demojize)

### Tokenization
Breaking sentences to usefull Words. Breaking such that their meaning don't change.

In [11]:
#Nltk not that accurate but for now, just use it
from nltk.tokenize import word_tokenize 
# nltk.download('punkt_tab')
df.clean_comment = df.clean_comment.apply(word_tokenize)
df.head()

,clean_comment,category
0,"[family, mormon, never, tried, explain, still,...",1
1,"[buddhism, much, lot, compatible, christianity...",1
2,"[seriously, say, thing, first, get, complex, e...",-1
3,"[learned, want, teach, different, focus, goal,...",0
4,"[benefit, may, want, read, living, buddha, liv...",1


#### Stemming

In [12]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
def stem_word(text_row):
    return (" ".join(ps.stem(word) for word in text_row))
df.clean_comment = df.clean_comment.apply(stem_word)
df.head(1)

,clean_comment,category
0,famili mormon never tri explain still stare pu...,1


## Tokens To vector
There are various algorithms- using varios libraries
1. Count based like BOW (lib: scikit-learn's countvectorizer), Tf-IDF etc.
2. Shallow neural net- Word2Vec algo (libraries implemented it- Gensim scratch algo implementaion)          
Spacy is another library with static vectorized words 
3. contextual Transformer based- BERT, GPT, SBERT etc.

In [13]:
#Gensim Library implementation of Word2Vec
import subprocess
import sys
import spacy
from spacy.util import is_package

MODEL = "en_core_web_sm"

if not is_package(MODEL):
    subprocess.check_call([
        sys.executable, "-m", "spacy", "download", MODEL
    ])

nlp = spacy.load(MODEL,disable=["tagger", "parser", "ner", "lemmatizer"])

def vectorize(text):
    return nlp(text).vector
    # vectors = [doc.vector for doc in nlp.pipe(text, batch_size=50, n_process=2)]
    # return vectors

df['vectorized_token'] = df.clean_comment.apply(vectorize)
df.head()

,clean_comment,category,vectorized_token
0,famili mormon never tri explain still stare pu...,1,"[0.33569604, -0.7258636, -0.059172656, -0.0716..."
1,buddhism much lot compat christian especi cons...,1,"[0.23283754, -0.5794332, 0.014645893, -0.01087..."
2,serious say thing first get complex explain no...,-1,"[0.06329079, -0.7204429, -0.07671744, -0.15280..."
3,learn want teach differ focu goal wrap paper b...,0,"[0.18831442, -0.72967166, 0.119656496, -0.1268..."
4,benefit may want read live buddha live christ ...,1,"[-0.055967435, -0.78625417, 0.0776148, -0.2207..."


#### Splitting of data should be done before then processing
1. The Correct OrderLoad Data: Get your raw data file.
2. Split Data: Divide into training and testing sets.
3. Preprocess Train Set: Fit your scalers and imputers on the training data only.
4. Transform Test Set: Apply those exact same training rules to transform the test data.

In [15]:
from sklearn.model_selection import train_test_split
x = df.iloc[:,2]
y = df.iloc[:,1]
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)    
x_train = np.reshape(x_train,shape=(-1, 1))
x_test = np.reshape(x_test,shape=(-1, 1))


train_lengths = np.array([len(row) for row in x_train.flatten()])
test_lengths = np.array([len(row) for row in x_test.flatten()])
mask = train_lengths == 96
mask2 = test_lengths == 96


x_train = np.stack(x_train.flatten()[mask])
x_test = np.stack(x_test.flatten()[mask2])

y_train = y_train[mask]
y_test = y_test[mask2]
print(f"Dropped {(~mask).sum()} rows")
print(f"Dropped {(~mask2).sum()} rows")

print(x_train.shape)
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)



Dropped 164 rows
Dropped 34 rows
(29555, 96)
x_train: (29555, 96)
x_test: (7396, 96)
y_train: (29555,)
y_test: (7396,)


#### Model training

In [28]:
ce = Transformer(x_train)
ypred,losshistory = ce.model.Fit(x_train,y_train,epoch=100,lr=0.005)

NameError: name 'Transformer' is not defined

In [ ]:
# from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
# #f1scrore, data we classified, how much was relavent?
# #recall, how much relavent data we classified?
# def evaluate_model(true, predicted):
#     accuracyscore = accuracy_score(true,predicted)
#     precisionscore = precision_score(true, predicted,average='micro')
#     recallscore = recall_score(true,predicted,average='micro')
#     f1score = f1_score(true,predicted,average='micro')
#     return accuracyscore, precisionscore, recallscore,f1score

In [ ]:
# from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
# from sklearn.ensemble import RandomForestClassifier
# fine_tune = [[20,30],[20,40],[20,50],[20,70],[20,100]] #best = [20,30]
# fine_tune2 = [[5,30],[10,30],[15,30],[20,30]] #best = no effect of change of min_samples_split

# # print(i[0],i[1])

# model = RandomForestClassifier(n_estimators=200,random_state=42,criterion='gini',min_samples_split=20, min_samples_leaf=5, max_depth=7,class_weight='balanced')
# model.fit(x_train,y_train)
# predict_train = model.predict(x_train)
# predict_test = model.predict(x_test)
# print("accuracy train",accuracy_score(y_train,predict_train,))
# print("precision train",precision_score(y_train,predict_train,average='weighted'))
# print("recall train",recall_score(y_train,predict_train,average='weighted'))
# print('-----Test')
# print("accuracy test",accuracy_score(y_train,predict_train,))
# print("precision test",precision_score(y_test,predict_test,average='weighted'))
# print("recall test",recall_score(y_test,predict_test,average='weighted'))
# print('--------------------End---------------------------------------')

# from sklearn.metrics import classification_report
# print(classification_report(y_test, predict_test))
# from sklearn.metrics import confusion_matrix
# import pandas as pd

# cm = confusion_matrix(y_test, predict_test, labels=[-1, 0, 1])
# print(pd.DataFrame(cm, index=[f'true_{l}' for l in [-1,0,1]], columns=[f'pred_{l}' for l in [-1,0,1]]))

NameError: name 'RandomForestClassifier' is not defined

In [ ]:
# from sklearn.preprocessing import LabelEncoder
# from xgboost import XGBClassifier
# from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

# # 1. Initialize and transform your target variable
# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

# fine_tune = [[20,30],[20,40],[20,50],[20,70],[20,100]] #best = [20,30]
# fine_tune2 = [[5,30],[10,30],[15,30],[20,30]] #best = no effect of change of min_samples_split

# # multiclass objective since categories are more than binary (-1,0,1)
# model = XGBClassifier(
# 	n_estimators=300,
# 	learning_rate=0.05,
# 	max_depth=3,
# 	objective='multi:softprob',
# 	num_class=len(le.classes_),
# 	use_label_encoder=False
# )

# model.early_stopping_rounds=10
# model.eval_metric='mlogloss',
# model.fit(
# 	x_train,
# 	y_train_encoded,
# 	verbose=False,
# 	eval_set = [(x_test, le.fit_transform(y_test))],
# )
# # model.evals_result(),

# predict_train = le.inverse_transform(model.predict(x_train))
# predict_test = le.inverse_transform(model.predict(x_test))

# print("accuracy train",accuracy_score(y_train,predict_train,))
# print("precision train",precision_score(y_train,predict_train,average='weighted'))
# print("recall train",recall_score(y_train,predict_train,average='weighted'))
# print('-----Test')
# print("accuracy test",accuracy_score(y_test,predict_test,))
# print("precision test",precision_score(y_test,predict_test,average='weighted'))
# print("recall test",recall_score(y_test,predict_test,average='weighted'))
# print('--------------------End---------------------------------------')

In [ ]:
# from sklearn.metrics import classification_report
# print(classification_report(y_test, predict_test))
# from sklearn.metrics import confusion_matrix
# import pandas as pd

# cm = confusion_matrix(y_test, predict_test, labels=[-1, 0, 1])
# print(pd.DataFrame(cm, index=[f'true_{l}' for l in [-1,0,1]], columns=[f'pred_{l}' for l in [-1,0,1]]))

              precision    recall  f1-score   support

          -1       0.57      0.29      0.38      1597
           0       0.74      0.74      0.74      2654
           1       0.64      0.80      0.71      3179

    accuracy                           0.67      7430
   macro avg       0.65      0.61      0.61      7430
weighted avg       0.66      0.67      0.65      7430

         pred_-1  pred_0  pred_1
true_-1      456     271     870
true_0       141    1970     543
true_1       207     435    2537


In [ ]:
# pos=df[df['category']==-1].sample(100)
# neg=df[df['category']==+1].sample(100)
# # print()
# scores =[]
# for i in range(100):
#     doc1= nlp(pos.clean_comment.iloc[i])
#     doc2= nlp(neg.clean_comment.iloc[i])
#     scores.append(doc1.similarity(doc2))
# scores = np.array(scores)
# (scores>0.5).sum()

/tmp/ipykernel_128454/1476665582.py:8: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  scores.append(doc1.similarity(doc2))


np.int64(75)

### Busted: Vector's been compromized
- Opposite sentiment vector contain similarity more then 60%, We can't rely on Spacy Now for vectors, that is why recall score for -1 was so low in evey model

### New Approach